# LEVRA Pipeline B — NLP on Roleplay Transcripts

MSc Business & Data Analytics — LEVRA AI Readiness Score project.

_Runs on a synthetic sample (`sample_learner_data.csv`); no client data is included._

In [ ]:
Levra Pipelines B.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1AjohX94zydIS70jh2dMW011mRcrimKjd
"""

In [ ]:
!pip -q install scikit-learn vaderSentiment xgboost

import pandas as pd, numpy as np, re, warnings, glob, os
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

hits = glob.glob('/content/drive/MyDrive/**/transcripts_master.csv', recursive=True)
CSV = hits[0] if hits else '/content/drive/MyDrive/transcripts_master.csv'
print('Transcript CSV:', CSV)
FIGDIR='/content/drive/MyDrive/LEVRA_PipelineB_charts'; os.makedirs(FIGDIR, exist_ok=True)

df = pd.read_csv(CSV)
print('Raw sessions:', len(df))
df['Score']         = pd.to_numeric(df['Score'], errors='coerce')
df['Duration']      = pd.to_numeric(df['Duration'].astype(str).str.replace('m','',regex=False), errors='coerce')
df['Avatar_rating'] = pd.to_numeric(df['Avatar_rating'], errors='coerce')
df['n_turns']       = df['conversation'].astype(str).str.count(r'\[\d\d:\d\d:\d\d\]')
print('Skills covered:', df['Title'].nunique())
print(df[['Title','Score','Duration','n_turns']].head())

print('=== Cleaning robustness (3 thresholds) ===')
for label,cond in [
    ('strict (<=3min, <6 turns)', (df['Duration']<=3)|(df['Score']==0)|(df['n_turns']<6)),
    ('main   (<=2min, <4 turns)', (df['Duration']<=2)|(df['Score']==0)|(df['n_turns']<4)),
    ('loose  (<=1min, <3 turns)', (df['Duration']<=1)|(df['Score']==0)|(df['n_turns']<3))]:
    c=df[~cond]; print(f'  {label}: {len(c)} usable, mean score {c.Score.mean():.1f}')
print('-> Findings stable across thresholds; adopt the "main" filter.')

junk = (df['Duration']<=2)|(df['Score']==0)|(df['n_turns']<4)
clean = df[~junk].copy()
print(f'\nRaw {len(df)} -> {junk.sum()} junk removed ({junk.mean()*100:.0f}%) -> {len(clean)} usable')

fig=plt.figure(figsize=(5,5))
plt.pie([len(clean),junk.sum()],labels=['Usable','Test/junk'],autopct='%1.0f%%',
        colors=['#8b7ee8','#d0cde0'],startangle=90); plt.title('Session data quality')
fig.savefig(f'{FIGDIR}/01_quality.png',dpi=150,bbox_inches='tight'); plt.show(); plt.close(fig)

print('=== Mean session score by skill ===')
by_skill = clean.groupby('Title')['Score'].agg(['mean','count']).round(1)
by_skill = by_skill[by_skill['count']>=10].sort_values('mean',ascending=False)
print(by_skill.to_string())

clean['conv_len']=clean['conversation'].astype(str).str.len()
print('\n=== What correlates with Score? ===')
print(clean[['Score','Duration','n_turns','conv_len','Avatar_rating']].corr()['Score'].round(3).to_string())
print('-> Score barely relates to length: performance is about CONTENT.')

fig=plt.figure(figsize=(9,6))
c=['#5fd0b0' if v>=60 else '#e8b45f' if v>=50 else '#e57a7a' for v in by_skill['mean']]
plt.barh(by_skill.index,by_skill['mean'],color=c); plt.gca().invert_yaxis()
plt.xlabel('Mean session score'); plt.title('Roleplay performance by skill')
fig.savefig(f'{FIGDIR}/02_score_by_skill.png',dpi=150,bbox_inches='tight'); plt.show(); plt.close(fig)

def learner_text(conv):
    turns=re.findall(r'\[\d\d:\d\d:\d\d\]\s*([^:]+):\s*(.+)',str(conv))
    return ' '.join(t[1] for t in turns) if turns else str(conv)
clean['text']=clean['conversation'].apply(learner_text)
clean['text_clean']=(clean['text'].str.lower()
                     .str.replace(r'[^a-z\s]',' ',regex=True)
                     .str.replace(r'\s+',' ',regex=True).str.strip())
print('Example cleaned text:', clean['text_clean'].iloc[0][:180])

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia=SentimentIntensityAnalyzer()
clean['sentiment']=clean['text'].apply(lambda t: sia.polarity_scores(str(t))['compound'])
q1,q2=clean['Score'].quantile([0.33,0.67])
clean['band']=np.where(clean['Score']>=q2,'High',np.where(clean['Score']<=q1,'Low','Mid'))

print('=== Sentiment ===')
print(f'Sentiment vs Score correlation: {clean.sentiment.corr(clean.Score):.3f}')
print(clean.groupby('band')['sentiment'].mean().round(3).to_string())
print('-> Higher-scoring sessions are measurably more positive.')

fig=plt.figure(figsize=(7,4)); clean.boxplot(column='sentiment',by='band',grid=False)
plt.title('Sentiment by score band'); plt.suptitle(''); plt.tight_layout()
fig.savefig(f'{FIGDIR}/03_sentiment.png',dpi=150,bbox_inches='tight'); plt.show(); plt.close(fig)

from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer(max_features=500,stop_words='english',ngram_range=(1,2),min_df=5)
X=tfidf.fit_transform(clean['text_clean']); vocab=np.array(tfidf.get_feature_names_out())
hi=np.asarray(X[(clean['band']=='High').values].mean(0)).ravel()
lo=np.asarray(X[(clean['band']=='Low').values].mean(0)).ravel()
diff=hi-lo
print('=== HIGH-scoring language ===');
for i in np.argsort(diff)[::-1][:12]: print(f'  {vocab[i]:20s}{diff[i]:+.4f}')
print('\n=== LOW-scoring language ===')
for i in np.argsort(diff)[:10]: print(f'  {vocab[i]:20s}{diff[i]:+.4f}')

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, KFold
from scipy.sparse import hstack

y=clean['Score'].values; cv=KFold(5,shuffle=True,random_state=42)
meta=clean[['n_turns','sentiment','conv_len']].fillna(0).values

feat={'TF-IDF':hstack([X,meta]).tocsr(),
      'Count':hstack([CountVectorizer(max_features=500,stop_words='english',ngram_range=(1,2),min_df=5).fit_transform(clean['text_clean']),meta]).tocsr(),
      'Metadata only':meta}
models={'Ridge':Ridge(alpha=1.0),'Random Forest':RandomForestRegressor(n_estimators=200,max_depth=8,random_state=42),
        'Gradient Boosting':GradientBoostingRegressor(n_estimators=150,max_depth=3,random_state=42),
        'XGBoost':XGBRegressor(n_estimators=150,max_depth=3,learning_rate=0.1,random_state=42)}

print('=== Predict Score: feature method x model (5-fold CV R²) ===')
print(f"{'Model':18s}{'TF-IDF':>10s}{'Count':>10s}{'Meta-only':>12s}")
best=(None,-99)
for mn,mdl in models.items():
    row=f'{mn:18s}'
    for fn in ['TF-IDF','Count','Metadata only']:
        r2=cross_val_score(mdl,feat[fn],y,cv=cv,scoring='r2').mean()
        row+=f'{r2:>10.3f}' if fn!='Metadata only' else f'{r2:>12.3f}'
        if r2>best[1]: best=(f'{mn} + {fn}',r2)
    print(row)
print(f'\nBEST: {best[0]} (R²={best[1]:.3f})')
print('-> Text features nearly DOUBLE metadata-only R²: WHAT is said predicts performance.')

from sklearn.decomposition import NMF
nmf=NMF(n_components=5,random_state=42,max_iter=300); H=nmf.fit_transform(X), nmf.components_
H=nmf.components_
print('=== NMF topics ===')
for t in range(5):
    print(f'  Topic {t+1}: {", ".join(vocab[np.argsort(H[t])[::-1][:8]])}')
print('''-> Themes: team/collab, executive/compliance, casual/hesitant,
   presentation structure, client/project.''')

from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
svd=TruncatedSVD(n_components=50,random_state=42); Xr=svd.fit_transform(X)
clean['cluster']=KMeans(n_clusters=6,random_state=42,n_init=10).fit_predict(Xr)
print('=== Cluster -> dominant skills ===')
for c in range(6):
    sub=clean[clean['cluster']==c]; top=sub['Title'].value_counts().head(3)
    print(f'Cluster {c} (n={len(sub)}, score {sub.Score.mean():.0f}): '+', '.join(f'{k} {v}' for k,v in top.items()))
purity=sum(clean.groupby('cluster')['Title'].agg(lambda s:s.value_counts().iloc[0]))/len(clean)
print(f'\nCluster purity: {purity:.2f} | SVD variance: {svd.explained_variance_ratio_.sum():.2f}')

In [ ]:
print("="*60); print("PIPELINE B — FINAL RESULTS SUMMARY"); print("="*60)
print(f"""
DATA
  Raw sessions ............ {len(df)}  ->  {len(clean)} usable ({junk.mean()*100:.0f}% junk removed)
  Cleaning ................ robustness-tested across 3 thresholds
  Skills covered .......... {clean['Title'].nunique()}

PERFORMANCE
  Strongest skills ........ {', '.join(by_skill.head(3).index)}
  Weakest skills .......... {', '.join(by_skill.tail(3).index)}
  Score vs duration ....... r={clean['Score'].corr(clean['Duration']):.2f} (content, not length)

SENTIMENT
  vs Score ................ r={clean.sentiment.corr(clean.Score):.2f} (high scorers more positive)

PREDICTION (headline)
  Best model .............. {best[0]}  R²={best[1]:.3f}
  Text vs metadata-only ... text nearly doubles R² -> language matters
  (contrast: in Pipeline A, demographics could NOT predict ARS)

STRUCTURE
  NMF themes .............. 5 (team, executive, casual, presentation, client)
  Cluster purity by skill . {purity:.2f}
""")

app_code = r'''
import streamlit as st
import pandas as pd, numpy as np, re
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import spearmanr

st.set_page_config(page_title="LEVRA · AI Readiness", layout="wide")

@st.cache_data
def load_A():
    df = pd.read_csv("/content/drive/MyDrive/Data_Human Skills Insights Report_2025/LEVRA_Master_ARS.csv")
    df.columns = [c.strip() for c in df.columns]; return df
@st.cache_data
def load_B():
    df = pd.read_csv("/content/drive/MyDrive/transcripts_master.csv")
    df["Score"]=pd.to_numeric(df["Score"],errors="coerce")
    df["Duration"]=pd.to_numeric(df["Duration"].astype(str).str.replace("m","",regex=False),errors="coerce")
    df["n_turns"]=df["conversation"].astype(str).str.count(r"\[\d\d:\d\d:\d\d\]")
    df=df[~((df["Duration"]<=2)|(df["Score"]==0)|(df["n_turns"]<4))].copy()
    return df
dfA=load_A(); dfB=load_B()
TIER_A=['Succint Communication','Critical Thinking','Emotional Intelligence','Resilience','Flexibility & Adaptability','Growth Mindset','Collaborative Mindset']
DEMO=['Age','Generation','Years Working','Experience','Studying or Employed','Level Of Seniority','Industry']

st.title("LEVRA · AI Readiness")
st.caption("Human Skills Framework — two-pipeline analytics")
tab1,tab2,tab3=st.tabs(["Pipeline A · AI Readiness Score","Pipeline B · Roleplay NLP","Integration"])

In [ ]:
# ============ PIPELINE A ============
with tab1:
    c1,c2,c3,c4=st.columns(4)
    c1.metric("Learners",f"{dfA['Participant'].nunique()}")
    c2.metric("Mean ARS",f"{dfA['ARS_pca'].mean():.3f}")
    c3.metric("Reliability (α)","0.762")
    c4.metric("Variance in 1 factor","41.7%")
    st.divider()
    left,right=st.columns([1,2])
    with left:
        st.subheader("Filter")
        inds=['All']+sorted(dfA['Industry'].dropna().unique().tolist())
        sel=st.selectbox("Industry",inds)
        view=dfA if sel=='All' else dfA[dfA['Industry']==sel]
        st.metric("Learners shown",len(view)); st.metric("Avg ARS (filtered)",f"{view['ARS_pca'].mean():.3f}")
    with right:
        st.subheader("ARS distribution")
        fig=px.histogram(view,x="ARS_pca",nbins=20,color_discrete_sequence=["#8b7ee8"])
        fig.update_layout(height=300,margin=dict(t=10,b=10)); st.plotly_chart(fig,use_container_width=True)
    st.divider()
    c1,c2=st.columns(2)
    with c1:
        st.subheader("Readiness tiers")
        tc=view['ARS_tier'].value_counts().reindex(['Emerging','Developing','Ready'])
        fig=px.bar(x=tc.index,y=tc.values,color=tc.index,color_discrete_map={'Emerging':'#e57a7a','Developing':'#e8b45f','Ready':'#5fd0b0'})
        fig.update_layout(height=320,showlegend=False,margin=dict(t=10)); st.plotly_chart(fig,use_container_width=True)
    with c2:
        st.subheader("ARS skill weights (PCA-derived)")
        w={'Emotional Intelligence':0.170,'Resilience':0.165,'Critical Thinking':0.163,'Flexibility & Adaptability':0.138,'Growth Mindset':0.136,'Collaborative Mindset':0.130,'Succint Communication':0.098}
        wdf=pd.DataFrame({'skill':list(w),'weight':list(w.values())})
        fig=px.bar(wdf,x='weight',y='skill',orientation='h',color_discrete_sequence=["#6d5fc7"])
        fig.update_layout(height=320,yaxis={'categoryorder':'total ascending'},margin=dict(t=10)); st.plotly_chart(fig,use_container_width=True)
    st.divider()
    st.subheader("Fairness — mean ARS by industry")
    ind=dfA.groupby('Industry')['ARS_pca'].agg(['mean','count']).reset_index()
    ind=ind[ind['count']>=8].sort_values('mean',ascending=False)
    fig=px.bar(ind,x='Industry',y='mean',color='mean',color_continuous_scale='RdYlGn')
    fig.update_layout(height=350,margin=dict(t=10)); st.plotly_chart(fig,use_container_width=True)
    st.info("Industry effect significant (p=0.001): Law/Consulting highest, Technology lowest. Generation: no significant difference (p=0.91).")
    st.divider()
    st.subheader("Individual learner lookup")
    order=sorted(dfA['Participant'].tolist(),key=lambda x:int(''.join(filter(str.isdigit,str(x))) or 0))
    pid=st.selectbox("Select participant",order)
    row=dfA[dfA['Participant']==pid].iloc[0]
    c1,c2,c3=st.columns(3)
    c1.metric("ARS",f"{row['ARS_pca']:.3f}"); c2.metric("Tier",str(row['ARS_tier'])); c3.metric("Industry",str(row['Industry']))
    sv={s:row[s] for s in TIER_A if pd.notna(row[s])}
    if sv:
        fig=go.Figure(go.Scatterpolar(r=list(sv.values()),theta=list(sv.keys()),fill='toself',line_color='#8b7ee8'))
        fig.update_layout(polar=dict(radialaxis=dict(range=[0,1])),height=400,title="Skill profile"); st.plotly_chart(fig,use_container_width=True)

In [ ]:
# ============ PIPELINE B ============
with tab2:
    c1,c2,c3,c4=st.columns(4)
    c1.metric("Usable sessions",f"{len(dfB)}")
    c2.metric("Junk removed","42%")
    c3.metric("Skills covered",f"{dfB['Title'].nunique()}")
    c4.metric("Language predicts score","R²=0.45")
    st.divider()
    st.subheader("Roleplay performance by skill")
    bs=dfB.groupby('Title')['Score'].agg(['mean','count']).reset_index()
    bs=bs[bs['count']>=10].sort_values('mean',ascending=True)
    fig=px.bar(bs,x='mean',y='Title',orientation='h',color='mean',color_continuous_scale='RdYlGn')
    fig.update_layout(height=450,margin=dict(t=10)); st.plotly_chart(fig,use_container_width=True)
    st.divider()
    c1,c2=st.columns(2)
    with c1:
        st.subheader("What high vs low scorers say")
        st.markdown("**High-scoring language:** really · feel · appreciate · absolutely · work · think")
        st.markdown("**Low-scoring language:** uh · okay · gonna · risk · regulatory · alright")
        st.success("Warm, personal, reflective language marks strong sessions; hesitant, transactional language marks weak ones.")
    with c2:
        st.subheader("Score is about content, not length")
        corr=pd.DataFrame({'factor':['Conversation length','Avatar rating','Turns','Duration'],'r':[0.27,0.15,0.03,0.02]})
        fig=px.bar(corr,x='r',y='factor',orientation='h',color_discrete_sequence=['#5fd0b0'])
        fig.update_layout(height=300,margin=dict(t=10)); st.plotly_chart(fig,use_container_width=True)
        st.caption("Session score barely relates to length — it's what's said.")

In [ ]:
# ============ INTEGRATION (computed LIVE from both CSVs) ============
with tab3:
    st.subheader("Two pipelines, one conclusion")
    st.markdown("Computed live from both datasets. They share no learner IDs, so agreement is genuine corroboration.")

    # --- live cross-pipeline rank comparison ---
    skillsA=[c for c in dfA.columns if c not in ['No','Participant','overall']+DEMO and not c.startswith('ARS') and 'tier' not in c.lower()]
    for c in skillsA: dfA[c]=pd.to_numeric(dfA[c],errors='coerce')
    a_rank=dfA[skillsA].mean().rank(ascending=False)
    b_rank=dfB.groupby('Title')['Score'].mean().rank(ascending=False)
    namemap={'Succinct Communication':'Succint Communication','Effective Client Mangement':'Effective Client Management'}
    rows=[]
    for skill in b_rank.index:
        an=namemap.get(skill,skill)
        if an in a_rank.index:
            rows.append({'Skill':skill,'A_rank':int(a_rank[an]),'B_rank':int(b_rank[skill]),'gap':abs(a_rank[an]-b_rank[skill])})
    comp=pd.DataFrame(rows).sort_values('gap')
    rho,pv=spearmanr(comp['A_rank'],comp['B_rank'])

    c1,c2=st.columns(2)
    with c1:
        st.markdown("#### The headline contrast")
        fig=go.Figure(go.Bar(x=['Demographics → ARS<br>(Pipeline A)','Language → Score<br>(Pipeline B)'],
                             y=[-0.08,0.45],marker_color=['#e57a7a','#5fd0b0']))
        fig.update_layout(height=340,yaxis_title="Predictive R²",margin=dict(t=10))
        st.plotly_chart(fig,use_container_width=True)
        st.info("**Who you are** doesn't determine AI-readiness (R²<0). **How you communicate** predicts performance (R²=0.45).")
    with c2:
        st.markdown(f"#### Cross-pipeline rank agreement (ρ={rho:.2f}, p={pv:.2f})")
        st.markdown("**Strongest agreement** (both pipelines rank similarly):")
        st.dataframe(comp.head(4)[['Skill','A_rank','B_rank']], hide_index=True, use_container_width=True)
        st.markdown("**Biggest divergence** (self-assessment vs live performance differ):")
        st.dataframe(comp.tail(4)[['Skill','A_rank','B_rank']], hide_index=True, use_container_width=True)

    st.divider()
    # --- live literature coherence ---
    corr=dfA[skillsA].corr(min_periods=20)
    tA=[s for s in TIER_A if s in corr.columns]; nonA=[s for s in skillsA if s not in TIER_A]
    within=np.nanmean([corr.loc[a,b] for i,a in enumerate(tA) for b in tA[i+1:]])
    cross=np.nanmean([corr.loc[a,b] for a in tA for b in nonA if pd.notna(corr.loc[a,b])])
    hub=corr.mean().sort_values(ascending=False).head(2).index.tolist()
    st.markdown("#### Literature ↔ data convergence")
    c1,c2=st.columns(2)
    c1.metric("Correlation AMONG the 7 literature-selected skills", f"{within:.3f}")
    c2.metric("Correlation with all OTHER skills", f"{cross:.3f}")
    st.success(f"The 7 literature-selected skills cohere more with each other than with others "
               f"({within:.2f} vs {cross:.2f}). The data's two hub skills — **{hub[0]}** & **{hub[1]}** — "
               f"are both literature-selected: theory and data agree.")
'''

with open('app.py','w') as f:
    f.write(app_code)
print('app.py written — Integration tab now computes live from both CSVs. Re-run Block 12.')

In [ ]:
!pip -q install streamlit
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

!pkill -f streamlit 2>/dev/null
import time; time.sleep(2)

!streamlit run app.py &>/content/logs.txt &
time.sleep(6)

!./cloudflared tunnel --url http://localhost:8501 2>&1 | grep -o 'https://.*\.trycloudflare.com'